In [3]:
import geopandas as gpd
import numpy as np
from mgwr.gwr import GWR
from mgwr.sel_bw import Sel_BW
import pandas as pd
from shapely.geometry import Point
from sklearn.preprocessing import StandardScaler

In [11]:
# Load CSV
df = pd.read_csv("Downloads/house_kmeans_pca.csv")

# Convert lat/lon to geometry points
geometry = [Point(xy) for xy in zip(df["Latitude"], df["Longitude"])]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

# Convert to UTM (for Philippines, UTM Zone 51N)
gdf = gdf.to_crs(epsg=32651)

# Define independent variables
independent_vars = ["LRTHubDist", "PC1", "PC2", "PC3", "PC4"]

# Extract independent variables and scale them
scaler = StandardScaler()
independent_scaled = scaler.fit_transform(gdf[independent_vars])

# Get dependent variable and scale it
dependent = gdf["Price"].values.reshape(-1, 1)
dependent_scaled = scaler.fit_transform(dependent)

# Get coordinates
coords = np.array(list(zip(gdf.geometry.x, gdf.geometry.y)))

# Select optimal bandwidth
bw = Sel_BW(coords, dependent_scaled, independent_scaled, kernel='gaussian', fixed=True).search(criterion='AICc')

# Fit GWR model
gwr_model = GWR(coords, dependent_scaled, independent_scaled, bw, kernel='gaussian', fixed=True)
results = gwr_model.fit()

# Print summary
print(results.summary())

# Add GWR coefficients to GeoDataFrame
for i, col in enumerate(independent_vars):  
    gdf[f"{col}_coef"] = results.params[:, i]

# Add R-squared values
gdf["Local_R2"] = results.localR2

# Save as Shapefile for QGIS
gdf.to_file("Downloads/GWR/gwr_pca_house_filtered.shp")

print("Shapefile saved as gwr_pca_house_filtered.shp. Load it into QGIS for visualization.")

Model type                                                         Gaussian
Number of observations:                                               33330
Number of covariates:                                                     6

Global Regression Results
---------------------------------------------------------------------------
Residual sum of squares:                                          25913.946
Log-likelihood:                                                  -43099.031
AIC:                                                              86210.061
AICc:                                                             86212.065
BIC:                                                            -321129.294
R2:                                                                   0.223
Adj. R2:                                                              0.222

Variable                              Est.         SE  t(Est/SE)    p-value
------------------------------- ---------- ---------- ------

C:\Users\Asus\AppData\Local\Temp\ipykernel_32692\1801435370.py:54: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file("Downloads/GWR/gwr_pca_house_filtered.shp")


Shapefile saved as gwr_pca_house_filtered.shp. Load it into QGIS for visualization.


C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'LRTHubDist_coef' to 'LRTHubDi_1'
  ogr_write(


In [5]:
def process_gwr(df, category, output_folder="Downloads/GWR"):

    # Convert lat/lon to geometry points
    geometry = [Point(xy) for xy in zip(df["Latitude"], df["Longitude"])]
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

    # Convert to UTM (for Philippines, UTM Zone 51N)
    gdf = gdf.to_crs(epsg=32651)

    # Define independent variables
    independent_vars = gdf[["LRTHubDist", "PC1", "PC2", "PC3", "PC4"]].values
    cols = ["LRTHubDist", "PC1", "PC2", "PC3", "PC4"]
    
    # Get dependent variable and scale it
    scaler = StandardScaler()
    dependent = gdf["Price"].values.reshape(-1, 1)
    dependent_scaled = scaler.fit_transform(dependent)

    # Get coordinates
    coords = np.array(list(zip(gdf.geometry.x, gdf.geometry.y)))

    # Select optimal bandwidth
    bw = Sel_BW(coords, dependent_scaled, independent_vars, kernel='gaussian', fixed=True).search(criterion='AICc')

    # Fit GWR model
    gwr_model = GWR(coords, dependent_scaled, independent_vars, bw, kernel='gaussian', fixed=True)
    results = gwr_model.fit()

    # Print summary
    print(f"\n--- GWR Summary for {category} ---")
    print(results.summary())

    # Add GWR coefficients to GeoDataFrame
    for i, col in enumerate(cols):  
        gdf[f"{col}_coef"] = results.params[:, i]

    # Add R-squared values
    gdf["Local_R2"] = results.localR2

    # Save as Shapefile
    output_file = f"{output_folder}/gwr_pca_{category}_filtered.shp"
    gdf.to_file(output_file)
    print(f"Shapefile saved: {output_file}")

In [5]:
df = pd.read_csv("Downloads/com_pca_tuned_dbscan.csv")  # Load each file
process_gwr(df, "com")    # Call function with loaded DataFrame


--- GWR Summary for com ---
Model type                                                         Gaussian
Number of observations:                                                2930
Number of covariates:                                                     6

Global Regression Results
---------------------------------------------------------------------------
Residual sum of squares:                                           2696.190
Log-likelihood:                                                   -4035.656
AIC:                                                               8083.313
AICc:                                                              8085.351
BIC:                                                             -20645.394
R2:                                                                   0.080
Adj. R2:                                                              0.078

Variable                              Est.         SE  t(Est/SE)    p-value
-------------------------------

C:\Users\Asus\AppData\Roaming\Python\Python312\site-packages\mgwr\gwr.py:775: RuntimeWarning: divide by zero encountered in divide
  return (self.TSS - self.RSS) / self.TSS
C:\Users\Asus\AppData\Roaming\Python\Python312\site-packages\mgwr\gwr.py:775: RuntimeWarning: invalid value encountered in divide
  return (self.TSS - self.RSS) / self.TSS


Shapefile saved: Downloads/GWR/gwr_pca_com_filtered.shp


C:\Users\Asus\AppData\Local\Temp\ipykernel_2468\684156348.py:42: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(output_file)
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'DBSCAN_Cluster' to 'DBSCAN_Clu'
  ogr_write(
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'LRTHubDist_coef' to 'LRTHubDi_1'
  ogr_write(
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value -6.2323897443118993e+155 of field Local_R2 of feature 1 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value -7.93440771506692206e+26 of field Local_R2 of feature 7 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(
C:\Users\As

In [7]:
df = pd.read_csv("Downloads/condo_pca_tuned_dbscan.csv")  # Load each file
process_gwr(df, "condo")    # Call function with loaded DataFrame


--- GWR Summary for condo ---
Model type                                                         Gaussian
Number of observations:                                               27266
Number of covariates:                                                     6

Global Regression Results
---------------------------------------------------------------------------
Residual sum of squares:                                          25932.786
Log-likelihood:                                                  -38005.322
AIC:                                                              76022.645
AICc:                                                             76024.649
BIC:                                                            -252484.383
R2:                                                                   0.049
Adj. R2:                                                              0.049

Variable                              Est.         SE  t(Est/SE)    p-value
-----------------------------

C:\Users\Asus\AppData\Local\Temp\ipykernel_2468\684156348.py:42: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(output_file)
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'DBSCAN_Cluster' to 'DBSCAN_Clu'
  ogr_write(
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'LRTHubDist_coef' to 'LRTHubDi_1'
  ogr_write(


Shapefile saved: Downloads/GWR/gwr_pca_condo_filtered.shp


In [8]:
df = pd.read_csv("Downloads/house_pca_tuned_dbscan.csv")  # Load each file
process_gwr(df, "house")    # Call function with loaded DataFrame


--- GWR Summary for house ---
Model type                                                         Gaussian
Number of observations:                                               33560
Number of covariates:                                                     6

Global Regression Results
---------------------------------------------------------------------------
Residual sum of squares:                                          26120.373
Log-likelihood:                                                  -43414.185
AIC:                                                              86840.370
AICc:                                                             86842.374
BIC:                                                            -323548.887
R2:                                                                   0.222
Adj. R2:                                                              0.222

Variable                              Est.         SE  t(Est/SE)    p-value
-----------------------------

C:\Users\Asus\AppData\Local\Temp\ipykernel_2468\684156348.py:42: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(output_file)
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'DBSCAN_Cluster' to 'DBSCAN_Clu'
  ogr_write(
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'LRTHubDist_coef' to 'LRTHubDi_1'
  ogr_write(


Shapefile saved: Downloads/GWR/gwr_pca_house_filtered.shp


In [9]:
df = pd.read_csv("Downloads/apt_birch_pca.csv")  # Load each file
process_gwr(df, "apt")    # Call function with loaded DataFrame


--- GWR Summary for apt ---
Model type                                                         Gaussian
Number of observations:                                                 304
Number of covariates:                                                     6

Global Regression Results
---------------------------------------------------------------------------
Residual sum of squares:                                            153.843
Log-likelihood:                                                    -327.831
AIC:                                                                667.662
AICc:                                                               670.040
BIC:                                                              -1549.831
R2:                                                                   0.494
Adj. R2:                                                              0.485

Variable                              Est.         SE  t(Est/SE)    p-value
-------------------------------

C:\Users\Asus\AppData\Local\Temp\ipykernel_2468\684156348.py:42: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(output_file)
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'LRTHubDist_coef' to 'LRTHubDi_1'
  ogr_write(


In [10]:
df = pd.read_csv("Downloads/land_birch_pca.csv")  # Load each file
process_gwr(df, "land")    # Call function with loaded DataFrame


--- GWR Summary for land ---
Model type                                                         Gaussian
Number of observations:                                                7959
Number of covariates:                                                     6

Global Regression Results
---------------------------------------------------------------------------
Residual sum of squares:                                           7946.499
Log-likelihood:                                                  -11287.076
AIC:                                                              22586.153
AICc:                                                             22588.167
BIC:                                                             -63487.813
R2:                                                                   0.002
Adj. R2:                                                              0.001

Variable                              Est.         SE  t(Est/SE)    p-value
------------------------------

C:\Users\Asus\AppData\Local\Temp\ipykernel_2468\684156348.py:42: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(output_file)
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'LRTHubDist_coef' to 'LRTHubDi_1'
  ogr_write(


Shapefile saved: Downloads/GWR/gwr_pca_land_filtered.shp


In [7]:
# Load the dataset
df = pd.read_csv("Downloads\GWR\Clustering\house_kmeans_pca.csv")

# Check unique clusters
unique_clusters = df["Cluster"].unique()

# Run process_gwr for each cluster
for cluster in unique_clusters:
    cluster_df = df[df["Cluster"] == cluster]  # Filter DataFrame for each cluster
    print(f"Processing Cluster {cluster} with {len(cluster_df)} rows...")
    process_gwr(cluster_df, f"house_cluster_{cluster}")  # Call function for each cluster

<>:2: SyntaxWarning: invalid escape sequence '\G'
<>:2: SyntaxWarning: invalid escape sequence '\G'
C:\Users\Asus\AppData\Local\Temp\ipykernel_8612\1798982914.py:2: SyntaxWarning: invalid escape sequence '\G'
  df = pd.read_csv("Downloads\GWR\Clustering\house_kmeans_pca.csv")


Processing Cluster 2 with 2353 rows...

--- GWR Summary for house_cluster_2 ---
Model type                                                         Gaussian
Number of observations:                                                2353
Number of covariates:                                                     6

Global Regression Results
---------------------------------------------------------------------------
Residual sum of squares:                                           1551.940
Log-likelihood:                                                   -2849.120
AIC:                                                               5710.240
AICc:                                                              5712.288
BIC:                                                             -16668.869
R2:                                                                   0.340
Adj. R2:                                                              0.339

Variable                              Est.         SE  t

C:\Users\Asus\AppData\Local\Temp\ipykernel_8612\684156348.py:42: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(output_file)
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'LRTHubDist_coef' to 'LRTHubDi_1'
  ogr_write(
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value -1.91921729916378338e+31 of field Local_R2 of feature 13 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(


Shapefile saved: Downloads/GWR/gwr_pca_house_cluster_2_filtered.shp
Processing Cluster 0 with 1717 rows...

--- GWR Summary for house_cluster_0 ---
Model type                                                         Gaussian
Number of observations:                                                1717
Number of covariates:                                                     6

Global Regression Results
---------------------------------------------------------------------------
Residual sum of squares:                                           1035.710
Log-likelihood:                                                   -2002.353
AIC:                                                               4016.706
AICc:                                                              4018.772
BIC:                                                             -11708.389
R2:                                                                   0.397
Adj. R2:                                                         

C:\Users\Asus\AppData\Local\Temp\ipykernel_8612\684156348.py:42: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(output_file)
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'LRTHubDist_coef' to 'LRTHubDi_1'
  ogr_write(
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value -2.96034213380648331e+23 of field Local_R2 of feature 1586 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(


Shapefile saved: Downloads/GWR/gwr_pca_house_cluster_0_filtered.shp
Processing Cluster 1 with 3509 rows...

--- GWR Summary for house_cluster_1 ---
Model type                                                         Gaussian
Number of observations:                                                3509
Number of covariates:                                                     6

Global Regression Results
---------------------------------------------------------------------------
Residual sum of squares:                                           1379.475
Log-likelihood:                                                   -3341.005
AIC:                                                               6694.011
AICc:                                                              6696.043
BIC:                                                             -27215.816
R2:                                                                   0.607
Adj. R2:                                                         

C:\Users\Asus\AppData\Local\Temp\ipykernel_8612\684156348.py:42: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(output_file)
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'LRTHubDist_coef' to 'LRTHubDi_1'
  ogr_write(


In [9]:
# Load the dataset
df = pd.read_csv("Downloads\GWR\Clustering\condo_kmeans_pca.csv")

# Check unique clusters
unique_clusters = df["Cluster"].unique()

# Run process_gwr for each cluster
for cluster in unique_clusters:
    cluster_df = df[df["Cluster"] == cluster]  # Filter DataFrame for each cluster
    print(f"Processing Cluster {cluster} with {len(cluster_df)} rows...")
    process_gwr(cluster_df, f"condo_cluster_{cluster}")  # Call function for each cluster

<>:2: SyntaxWarning: invalid escape sequence '\G'
<>:2: SyntaxWarning: invalid escape sequence '\G'
C:\Users\Asus\AppData\Local\Temp\ipykernel_8612\2591409660.py:2: SyntaxWarning: invalid escape sequence '\G'
  df = pd.read_csv("Downloads\GWR\Clustering\condo_kmeans_pca.csv")


Processing Cluster 0 with 1828 rows...

--- GWR Summary for condo_cluster_0 ---
Model type                                                         Gaussian
Number of observations:                                                1828
Number of covariates:                                                     6

Global Regression Results
---------------------------------------------------------------------------
Residual sum of squares:                                            828.083
Log-likelihood:                                                   -1870.056
AIC:                                                               3752.111
AICc:                                                              3754.173
BIC:                                                             -12856.919
R2:                                                                   0.547
Adj. R2:                                                              0.546

Variable                              Est.         SE  t

C:\Users\Asus\AppData\Local\Temp\ipykernel_8612\684156348.py:42: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(output_file)
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'LRTHubDist_coef' to 'LRTHubDi_1'
  ogr_write(


Shapefile saved: Downloads/GWR/gwr_pca_condo_cluster_0_filtered.shp
Processing Cluster 2 with 1672 rows...

--- GWR Summary for condo_cluster_2 ---
Model type                                                         Gaussian
Number of observations:                                                1672
Number of covariates:                                                     6

Global Regression Results
---------------------------------------------------------------------------
Residual sum of squares:                                            496.614
Log-likelihood:                                                   -1357.592
AIC:                                                               2727.184
AICc:                                                              2729.251
BIC:                                                             -11868.065
R2:                                                                   0.703
Adj. R2:                                                         

C:\Users\Asus\AppData\Local\Temp\ipykernel_8612\684156348.py:42: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(output_file)
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'LRTHubDist_coef' to 'LRTHubDi_1'
  ogr_write(
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value -4.87885707929298313e+148 of field Local_R2 of feature 384 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value -2.49935195882028456e+29 of field Local_R2 of feature 706 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value -3.50971492211177333e+70 of field Local_R2 of feature 1156 not successfully wri

Shapefile saved: Downloads/GWR/gwr_pca_condo_cluster_2_filtered.shp
Processing Cluster 1 with 3406 rows...

--- GWR Summary for condo_cluster_1 ---
Model type                                                         Gaussian
Number of observations:                                                3406
Number of covariates:                                                     6

Global Regression Results
---------------------------------------------------------------------------
Residual sum of squares:                                           1356.945
Log-likelihood:                                                   -3265.629
AIC:                                                               6543.257
AICc:                                                              6545.290
BIC:                                                             -26296.254
R2:                                                                   0.602
Adj. R2:                                                         

C:\Users\Asus\AppData\Local\Temp\ipykernel_8612\684156348.py:42: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(output_file)
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'LRTHubDist_coef' to 'LRTHubDi_1'
  ogr_write(


In [19]:
# Load the dataset
df = pd.read_csv(r"Downloads\GWR\Clustering\apt_kmeans_pca.csv")

# Check unique clusters
unique_clusters = df["Cluster"].unique()

# Run process_gwr for each cluster
for cluster in unique_clusters:
    cluster_df = df[df["Cluster"] == cluster]  # Filter DataFrame for each cluster
    print(f"Processing Cluster {cluster} with {len(cluster_df)} rows...")
    process_gwr(cluster_df, f"apt_cluster_{cluster}")  # Call function for each cluster

Processing Cluster 0 with 22 rows...

--- GWR Summary for apt_cluster_0 ---
Model type                                                         Gaussian
Number of observations:                                                  22
Number of covariates:                                                     6

Global Regression Results
---------------------------------------------------------------------------
Residual sum of squares:                                             13.837
Log-likelihood:                                                     -26.116
AIC:                                                                 64.232
AICc:                                                                74.232
BIC:                                                                -35.620
R2:                                                                   0.371
Adj. R2:                                                              0.175

Variable                              Est.         SE  t(Est

C:\Users\Asus\AppData\Local\Temp\ipykernel_8612\684156348.py:42: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(output_file)
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'LRTHubDist_coef' to 'LRTHubDi_1'
  ogr_write(



--- GWR Summary for apt_cluster_2 ---
Model type                                                         Gaussian
Number of observations:                                                  11
Number of covariates:                                                     6

Global Regression Results
---------------------------------------------------------------------------
Residual sum of squares:                                              0.201
Log-likelihood:                                                       6.407
AIC:                                                                 -0.814
AICc:                                                                38.520
BIC:                                                                -11.789
R2:                                                                   0.982
Adj. R2:                                                              0.963

Variable                              Est.         SE  t(Est/SE)    p-value
---------------------

C:\Users\Asus\AppData\Local\Temp\ipykernel_8612\684156348.py:42: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(output_file)
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'LRTHubDist_coef' to 'LRTHubDi_1'
  ogr_write(



--- GWR Summary for apt_cluster_4 ---
Model type                                                         Gaussian
Number of observations:                                                  19
Number of covariates:                                                     6

Global Regression Results
---------------------------------------------------------------------------
Residual sum of squares:                                              3.985
Log-likelihood:                                                     -12.122
AIC:                                                                 36.243
AICc:                                                                48.425
BIC:                                                                -34.293
R2:                                                                   0.790
Adj. R2:                                                              0.710

Variable                              Est.         SE  t(Est/SE)    p-value
---------------------

C:\Users\Asus\AppData\Local\Temp\ipykernel_8612\684156348.py:42: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(output_file)
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'LRTHubDist_coef' to 'LRTHubDi_1'
  ogr_write(



--- GWR Summary for apt_cluster_3 ---
Model type                                                         Gaussian
Number of observations:                                                  28
Number of covariates:                                                     6

Global Regression Results
---------------------------------------------------------------------------
Residual sum of squares:                                             16.674
Log-likelihood:                                                     -32.473
AIC:                                                                 76.946
AICc:                                                                84.546
BIC:                                                                -56.635
R2:                                                                   0.405
Adj. R2:                                                              0.269

Variable                              Est.         SE  t(Est/SE)    p-value
---------------------

C:\Users\Asus\AppData\Local\Temp\ipykernel_8612\684156348.py:42: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(output_file)
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'LRTHubDist_coef' to 'LRTHubDi_1'
  ogr_write(


ValueError: array must not contain infs or NaNs

In [23]:
# Load the dataset
df = pd.read_csv(r"Downloads\GWR\Clustering\com_pca_tuned_dbscan.csv")

# Check unique clusters
unique_clusters = df["DBSCAN_Cluster"].unique()

# Run process_gwr for each cluster
for cluster in unique_clusters:
    cluster_df = df[df["DBSCAN_Cluster"] == cluster]  # Filter DataFrame for each cluster
    
    # Skip if the number of observations is less than 6
    if len(cluster_df) < 6:
        print(f"Skipping Cluster {cluster} (only {len(cluster_df)} rows)...")
        continue
    
    print(f"Processing Cluster {cluster} with {len(cluster_df)} rows...")
    process_gwr(cluster_df, f"com_cluster_{cluster}")  # Call function for each cluster

Processing Cluster 0 with 453 rows...

--- GWR Summary for com_cluster_0 ---
Model type                                                         Gaussian
Number of observations:                                                 453
Number of covariates:                                                     6

Global Regression Results
---------------------------------------------------------------------------
Residual sum of squares:                                            225.239
Log-likelihood:                                                    -484.517
AIC:                                                                981.033
AICc:                                                               983.285
BIC:                                                              -2508.565
R2:                                                                   0.503
Adj. R2:                                                              0.497

Variable                              Est.         SE  t(Es

C:\Users\Asus\AppData\Local\Temp\ipykernel_8612\684156348.py:42: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(output_file)
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'DBSCAN_Cluster' to 'DBSCAN_Clu'
  ogr_write(
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'LRTHubDist_coef' to 'LRTHubDi_1'
  ogr_write(



--- GWR Summary for com_cluster_1 ---
Model type                                                         Gaussian
Number of observations:                                                 515
Number of covariates:                                                     6

Global Regression Results
---------------------------------------------------------------------------
Residual sum of squares:                                            250.841
Log-likelihood:                                                    -545.522
AIC:                                                               1103.044
AICc:                                                              1105.265
BIC:                                                              -2927.439
R2:                                                                   0.513
Adj. R2:                                                              0.508

Variable                              Est.         SE  t(Est/SE)    p-value
---------------------

C:\Users\Asus\AppData\Roaming\Python\Python312\site-packages\mgwr\gwr.py:775: RuntimeWarning: divide by zero encountered in divide
  return (self.TSS - self.RSS) / self.TSS
C:\Users\Asus\AppData\Local\Temp\ipykernel_8612\684156348.py:42: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(output_file)
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'DBSCAN_Cluster' to 'DBSCAN_Clu'
  ogr_write(
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'LRTHubDist_coef' to 'LRTHubDi_1'
  ogr_write(
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value -1.89052031019371821e+102 of field Local_R2 of feature 3 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: 

In [25]:
# Load the dataset
df = pd.read_csv("Downloads\GWR\Clustering\land_kmeans_pca.csv")

# Check unique clusters
unique_clusters = df["Cluster"].unique()

# Run process_gwr for each cluster
for cluster in unique_clusters:
    cluster_df = df[df["Cluster"] == cluster]  # Filter DataFrame for each cluster
    print(f"Processing Cluster {cluster} with {len(cluster_df)} rows...")
    process_gwr(cluster_df, f"land_cluster_{cluster}")  # Call function for each cluster

<>:2: SyntaxWarning: invalid escape sequence '\G'
<>:2: SyntaxWarning: invalid escape sequence '\G'
C:\Users\Asus\AppData\Local\Temp\ipykernel_8612\3973211268.py:2: SyntaxWarning: invalid escape sequence '\G'
  df = pd.read_csv("Downloads\GWR\Clustering\land_kmeans_pca.csv")


Processing Cluster 0 with 878 rows...

--- GWR Summary for land_cluster_0 ---
Model type                                                         Gaussian
Number of observations:                                                 878
Number of covariates:                                                     6

Global Regression Results
---------------------------------------------------------------------------
Residual sum of squares:                                            510.317
Log-likelihood:                                                   -1007.620
AIC:                                                               2027.240
AICc:                                                              2029.369
BIC:                                                              -5399.791
R2:                                                                   0.419
Adj. R2:                                                              0.415

Variable                              Est.         SE  t(E

C:\Users\Asus\AppData\Local\Temp\ipykernel_8612\684156348.py:42: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(output_file)
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'LRTHubDist_coef' to 'LRTHubDi_1'
  ogr_write(


Shapefile saved: Downloads/GWR/gwr_pca_land_cluster_0_filtered.shp
Processing Cluster 2 with 466 rows...

--- GWR Summary for land_cluster_2 ---
Model type                                                         Gaussian
Number of observations:                                                 466
Number of covariates:                                                     6

Global Regression Results
---------------------------------------------------------------------------
Residual sum of squares:                                            337.597
Log-likelihood:                                                    -586.122
AIC:                                                               1184.244
AICc:                                                              1186.489
BIC:                                                              -2488.728
R2:                                                                   0.276
Adj. R2:                                                            

C:\Users\Asus\AppData\Local\Temp\ipykernel_8612\684156348.py:42: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(output_file)
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'LRTHubDist_coef' to 'LRTHubDi_1'
  ogr_write(



--- GWR Summary for land_cluster_1 ---
Model type                                                         Gaussian
Number of observations:                                                 753
Number of covariates:                                                     6

Global Regression Results
---------------------------------------------------------------------------
Residual sum of squares:                                            378.041
Log-likelihood:                                                    -809.029
AIC:                                                               1630.058
AICc:                                                              1632.208
BIC:                                                              -4570.135
R2:                                                                   0.498
Adj. R2:                                                              0.495

Variable                              Est.         SE  t(Est/SE)    p-value
--------------------

C:\Users\Asus\AppData\Local\Temp\ipykernel_8612\684156348.py:42: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(output_file)
C:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'LRTHubDist_coef' to 'LRTHubDi_1'
  ogr_write(
